In [5]:
!pip install azure-storage-blob openai
!pip install azure-ai-projects openai azure-identity


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\acane\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\acane\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import json
import gzip
import io
import sys
from azure.storage.blob import BlobServiceClient
from openai import AzureOpenAI

# --- CONFIGURATION ---
ACCOUNT_URL = "https://aegisscholardata.blob.core.windows.net/"
SAS_TOKEN = "?sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupiytfx&se=2026-05-30T08:31:15Z&st=2026-02-12T01:16:15Z&spr=https&sig=pU5C5a%2B%2BxE1zvMMv3vjUqjlJXC9dMgpsVyM0V%2FfuEIo%3D" 
SOURCE_CONTAINER = "raw"
BLOB_PREFIX = "dtic/"

AZURE_ENDPOINT = "https://acanedo137-2668-resource.cognitiveservices.azure.com/"
AZURE_KEY = "FabENbzRtxgDTLp89JypC2Ae4IT0jk8L9kS6MfcyVyIoXGHE8RTdJQQJ99CBACHYHv6XJ3w3AAAAACOGvIme"
DEPLOYMENT_NAME = "gpt-4o" 
API_VERSION = "2025-01-01-preview"

# --- CACHE & STATS ---
local_cache = {}

# Initialize Clients
blob_service_client = BlobServiceClient(account_url=ACCOUNT_URL, credential=SAS_TOKEN)
llm_client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_KEY, api_version=API_VERSION)

def get_llm_contact_info(org_name, country):
    """Runs the org name against the LLM if not in local cache."""
    cache_key = f"{org_name}_{country}".lower().strip()
    
    if cache_key in local_cache:
        return local_cache[cache_key], True
        
    prompt = (
        f"Find the official public contact email and phone number for the organization '{org_name}' "
        f"located in '{country or 'Unknown'}'. "
        "Return strictly valid JSON with keys 'email' and 'phone'. Use null if you cannot find them."
    )
    
    try:
        response = llm_client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0
        )
        contact_info = json.loads(response.choices[0].message.content)
        local_cache[cache_key] = contact_info
        return contact_info, False
    except Exception as e:
        return {"email": None, "phone": None}, False

def decompress_and_read(blob_client):
    """Downloads and reads the blob, handling gzip if necessary."""
    content = blob_client.download_blob().readall()
    if content.startswith(b'\x1f\x8b'):
        with gzip.GzipFile(fileobj=io.BytesIO(content)) as f:
            return f.read().decode('utf-8', errors='replace')
    return content.decode('utf-8', errors='replace')

def parse_records(raw_text):
    """Smartly parses the text whether it's a JSON array, single dict, or JSON Lines."""
    # 1. Try parsing the whole file as one JSON structure
    try:
        data = json.loads(raw_text)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            return [data] # FIX: If it's a single dictionary, wrap it in a list!
    except json.JSONDecodeError:
        pass
        
    # 2. If it fails, parse it line by line (JSONL)
    records = []
    for line in raw_text.splitlines():
        clean_line = line.strip()
        if not clean_line:
            continue
        try:
            parsed = json.loads(clean_line)
            # Double-encoding check
            if isinstance(parsed, str):
                parsed = json.loads(parsed)
            records.append(parsed)
        except Exception:
            continue
    return records

def main():
    container = blob_service_client.get_container_client(SOURCE_CONTAINER)
    print(f"🔍 Searching in container '{SOURCE_CONTAINER}' for prefix '{BLOB_PREFIX}'...")
    
    try:
        blobs = list(container.list_blobs(name_starts_with=BLOB_PREFIX))
    except Exception as e:
        print(f"❌ Failed to access blob storage: {e}")
        sys.exit(1)
        
    if not blobs:
        print("❌ Found 0 files! Please check your BLOB_PREFIX and SAS_TOKEN.")
        sys.exit(1)
        
    print(f"✅ Found {len(blobs)} files. Starting extraction...\n")
    
    final_results = []
    org_count = 0
    TARGET_COUNT = 500

    for blob in blobs:
        if org_count >= TARGET_COUNT:
            break
            
        # print(f"📂 Reading file: {blob.name}") # Optional: comment this out if it clutters the screen
        raw_text = decompress_and_read(container.get_blob_client(blob))
        records = parse_records(raw_text)
        
        for record in records:
            if org_count >= TARGET_COUNT:
                break
            
            # THE SAFETY SHIELD: Only process if it is strictly a dictionary
            if not isinstance(record, dict):
                continue
                
            orgs = record.get("organizations", [])
            
            # Sometimes 'organizations' might be accidentally saved as a stringified list in the DB
            if isinstance(orgs, str):
                try: orgs = json.loads(orgs)
                except: orgs = []

            # Safety check: ensure orgs is actually a list before we loop
            if not isinstance(orgs, list):
                continue

            for org in orgs:
                if org_count >= TARGET_COUNT:
                    break
                
                # Double-check that the organization item itself is a dict
                if not isinstance(org, dict):
                    continue
                    
                org_name = org.get("name")
                org_country = org.get("country")
                
                if org_name:
                    org_count += 1
                    contact_info, from_cache = get_llm_contact_info(org_name, org_country)
                    
                    status = "⚡ CACHED" if from_cache else "🤖 LLM   "
                    print(f"[{org_count:03d}/{TARGET_COUNT}] {status} | {org_name[:40]} ({org_country})")
                    
                    final_results.append({
                        "id": org_count,
                        "publication_id": record.get("publication_id"),
                        "organization_name": org_name,
                        "country": org_country,
                        "contact_email": contact_info.get("email"),
                        "contact_phone": contact_info.get("phone")
                    })

    with open("dtic_enriched_organizations.json", "w", encoding="utf-8") as f:
        json.dump(final_results, f, indent=4, ensure_ascii=False)

    print(f"\n🎉 Extraction complete! Saved {org_count} organizations to 'dtic_enriched_organizations.json'.")

if __name__ == "__main__":
    main()

🔍 Searching in container 'raw' for prefix 'dtic/'...
✅ Found 28071 files. Starting extraction...

[001/500] 🤖 LLM    | Georgia Institute of Technology (United States)
[002/500] 🤖 LLM    | Brunel University London (United Kingdom)
[003/500] 🤖 LLM    | Kansas State University (United States)
[004/500] 🤖 LLM    | Roswell Park Comprehensive Cancer Center (United States)
[005/500] 🤖 LLM    | University of Minnesota Twin Cities (United States)
[006/500] 🤖 LLM    | University of Bristol (United Kingdom)
[007/500] 🤖 LLM    | University of Groningen (Netherlands)
[008/500] 🤖 LLM    | The University of Texas MD Anderson Canc (United States)
[009/500] 🤖 LLM    | Medical College of Wisconsin (United States)
[010/500] 🤖 LLM    | Ankara University (Turkey)
[011/500] 🤖 LLM    | Heidelberg University (Germany)
[012/500] 🤖 LLM    | Instituto Scientifico HS Raffaele, Milan (Italy)
[013/500] 🤖 LLM    | Ospedale Policlinico San Martino (Italy)
[014/500] 🤖 LLM    | Institute Paoli-Calmettes (France)
[015/5